# Parallel Thermosyphon-Driven flow


In this example we'll demonstrate how to build parallel hydraulic loops driven by temprature differences in each vertical branch, which induce thermosyphonic flow. 
For the sake of simplicity we'll consider only the Kirchhoff equations without energy conservation.

Assuming the case of three hydraulic resistors connected in parallel in vertical channels. each with different temprature (hot, mean and cold). 
This serves as a test case for inner-loop circulations that can happen in thermosyphon driven flow in reactor cores with channels at different temperatures.

$$
\begin{align}
\dot{m}_c&=\dot{m}_h+\dot{m}_m \\
\dot{m}_cK_c+\dot{m}_hK_h&=(\rho_c-\rho_h)g\Delta h \\
\dot{m}_cK_c+\dot{m}_mK_m&=(\rho_c-\rho_m)g\Delta h
\end{align}
$$

which can be simplified into:

$$
\begin{equation}
\begin{pmatrix}
K_c+K_h & K_c \\
K_c &K_c+K_m
\end{pmatrix}
\begin{pmatrix}
\dot{m}_h \\
\dot{m}_m
\end{pmatrix}=g\Delta h
\begin{pmatrix}
\rho_c-\rho_h \\
\rho_c-\rho_m
\end{pmatrix}
\end{equation}
$$



All that's left is to invert the matrix and we'll obtain the mass flow rates through each hydraulic resistor.
This can be done by hand.

The solution is:
$$
\begin{align}
\dot{m}_h &= \Delta hg\frac{(K_c+K_m)(\rho_c-\rho_h) - K_c(\rho_c-\rho_m)}{K_cK_h+K_cK_m+K_hK_m} \\
\dot{m}_m &= \Delta hg\frac{(K_c+K_h)(\rho_c-\rho_m) - K_c(\rho_c-\rho_h)}{K_cK_h+K_cK_m+K_hK_m} \\
\end{align}
$$

In [ ]:
def analytic_solution(dh, g, kc, kh, km, rc, rh, rm):
    denominator = kc * kh + kc * km + kh * km
    mh = dh * g * ((kc + km) * (rc - rh) - kc * (rc - rm)) / denominator
    mm = dh * g * ((kc + kh) * (rc - rm) - kc * (rc - rh)) / denominator
    return mh, mm

After we obtained the solution analytiaclly, we'll solve this problem using the `stream` library.

The first step is to import the relevant objects, representing hydraulic resistors, gravity, junctions, flow graph etc.

In [ ]:
from functools import partial

import numpy as np

from stream.calculations import Gravity, Resistor
from stream.calculations.kirchhoff import Junction
from stream.composition import flow_edge, flow_graph, flow_graph_to_agr_and_k
from stream.substances import light_water as h2o
from stream.units import g

We will also need to use actual numbers for the numeric solution:

In [ ]:
Tc, Th, Tm = 20.0, 50.0, 40.0
rc, rh, rm = [h2o.density(t) for t in [Tc, Th, Tm]]
kc, kh, km = 1, 10, 10
dh = 1

In our problem we need to create 2 junction objects, which we name "top" and "bottom".

In [ ]:
top = Junction("top")
bottom = Junction("bottom")

for functions and classes that require the same repeated inputs we can use `functools.partial()` function to create a partially applied function that is imbued with the repeating inputs.

In [ ]:
gravity = partial(Gravity, fluid=h2o, gravity=g)

Now we can create the hydraulic circuit using `flow_graph` where each branch is defined using `flow_edge`.

This is the basic way to represent hydraulic relations in `Stream`. It's important to notice that the hyraulic circuit is closed, so Kirchhoff calculatiosn can do closed loop equations. Hence, for the example below, if you defined `flow_edge` that started from a `junction` named `top` you have to define a `flow_edge` returning to `top`. Another important point here is the sign of `Gravity`'s object disposition $\Delta h$. In `Stream` it should be positive for downard flow (in the direction of gravity) and negative for upward flow. The flow rates are positive when the flow is in the same direction as the appropriate flow edge.

In [ ]:
fg = flow_graph(
    flow_edge((top, bottom), Resistor(kh, "Rh"), gravity(disposition=dh, name="gh")),
    flow_edge((top, bottom), Resistor(km, "Rm"), gravity(disposition=dh, name="gm")),
    flow_edge((bottom, top), Resistor(kc, "Rc"), gravity(disposition=-dh, name="gc")),
)

The next step is to build an aggregator of the equations out of the `flow_graph` we constructed, which can be done using `flow_graph_to_agr_k`.

In [ ]:
agr, _ = flow_graph_to_agr_and_k(fg)

All that's left before solving the problem is to define the temperatures at each branch. 
Because we only look at momentum conservation and not really intrested in energy conservation in this toy problem, it's enough to define the temperatures of each gravity object using the concept of `ext_funcs`.

In [ ]:
agr.funcs = {
    agr["gh"]: {"Tin": Th, "Tin_minus": Th},
    agr["gm"]: {"Tin": Tm, "Tin_minus": Tm},
    agr["gc"]: {"Tin": Tc, "Tin_minus": Tc},
}

A very nice tool to inspect the number of equations and variables is the `report` function. This will tell us if something is not properlly defined or missing before trying to solve the agregator.

In [ ]:
from stream.analysis.report import report

report(agr)

Finally we'll solve the problem using the `agr.solve_steady()` method with an initial guess of a vector of ones. 
We also use `agr.save()` to transform the solution from a vector to a dictionary with keys that indicate the context of each value in the result vector.

In [ ]:
stream_sol = agr.save(agr.solve_steady(np.ones(len(agr)), options={"xtol": 1e-13}))

The analytic function should also be computed for our specific values:

In [ ]:
analytic_sol = analytic_solution(dh, g, kc, kh, km, rc, rh, rm)

Then we can show what the solution is and `stream`'s error:

In [ ]:
mdot_h_stream = stream_sol["Kirchhoff"]["(top -> bottom, 0)"]
mdot_h_analytical = -analytic_sol[0]
print(f"mdot_h={mdot_h_analytical:.3e}\nerror:{100 * (mdot_h_analytical - mdot_h_stream) / mdot_h_analytical:.5e} %")

In [ ]:
mdot_m_stream = stream_sol["Kirchhoff"]["(top -> bottom, 1)"]
mdot_m_analytical = -analytic_sol[1]
print(f"mdot_m={mdot_m_analytical:.3e}\nerror:{100 * (mdot_m_analytical - mdot_m_stream) / mdot_m_analytical:.5e} %")

 Thus the numeric result obtained is really close (numerically identical) to the analytical solution